# Exercise 1 - PydanticAI, FastAPI and LLM judge

0. PydanticAI fundamentals


Make a PydanticAI model that can take an input of a location and then it should suggest 5 restaurants nearby that place. The restaurant model should have

- name
- type of food (cuisine)
- price level
- rating
- short description
- opening hours
- location

#### Pydantic model & created agent

In [21]:
from pydantic import BaseModel, Field
from typing import Literal 
from pydantic_ai import Agent

class RestaurantSearcher(BaseModel):
    name: str = Field(description="Restaurant name")
    type_of_food: str = Field(description="Type of food or cuisine")
    price_level: Literal["$","$$","$$$"] = Field(description="Approxiately price range")
    rating: Literal["1","2","3","4","5"] 
    short_description: str = Field(description="Short summary about the type of cusinie the Restaurant offers")
    opening_hours: float = Field(descrioption="Opening hours as text")
    location: str = Field(description="Restaruant location/address/area")

class RestaurantList(BaseModel):
    restaurants: list[RestaurantSearcher] = Field(
        min_length=5,
        max_length=5,
        description="Exactly 5 restaurant suggestions"
    )

restaurant_searcher_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=""" 
You are a restaurant recommendation assistant.
The user gives you a location.
Return exactly 5 restaurant near the place.
It is allowed to invent restaurant if needed, 
but they should still look realistic and relevant to the location.
Very cuisine types.
Keep descriptions short and useful.
Ratings must be between 1 and 5.
Price level must be one of: $, $$, $$$


    """, output_type=RestaurantList)



C:\Users\tasmi\AppData\Local\Temp\ipykernel_23160\1935766385.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descrioption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  opening_hours: float = Field(descrioption="Opening hours as text")


In [23]:
location = "Södermalm, Stockholm"

result = await restaurant_searcher_agent.run(location)

result.output

RestaurantList(restaurants=[RestaurantSearcher(name='Köttbullar & Co', type_of_food='Swedish meatballs', price_level='$$', rating='4', short_description='Classic Swedish meatballs with lingonberry sauce and mashed potatoes.', opening_hours=12.0, location='Götgatan 34, Södermalm, Stockholm'), RestaurantSearcher(name='Söder Sushi', type_of_food='Japanese sushi', price_level='$$$', rating='5', short_description='Fresh sushi rolls and sashimi with a modern twist.', opening_hours=10.0, location='Hornsgatan 78, Södermalm, Stockholm'), RestaurantSearcher(name='La Piazza', type_of_food='Italian pasta', price_level='$$', rating='4', short_description='Handmade pasta dishes and wood-fired pizzas in a cozy setting.', opening_hours=11.0, location='Skånegatan 55, Södermalm, Stockholm'), RestaurantSearcher(name='Green Leaf Veggie', type_of_food='Vegan/Vegetarian', price_level='$', rating='4', short_description='Creative plant-based bowls, burgers, and smoothies.', opening_hours=9.0, location='Mariat

In [36]:
async def get_restaurant(location: str) -> RestaurantList:
    prompt = f"Suggest 5 restaurants near {location}."
    result = restaurant_searcher_agent.run(prompt)
    return result.output